In [13]:
from typing import Literal, Callable
import numpy as np
import polars as pl

In [14]:
EPSILON = 1e-12

In [15]:
def wasserstein_transition_distance_to_optimal(
    Q: np.ndarray,
    biased: Literal["good proposals", "bad proposals", "weight on one", False] = False,
) -> float:
    rows, cols = Q.shape

    match biased:
        case False:
            weights = np.ones(cols) / cols
        case "good proposals":
            weights = np.arange(1, cols + 1)[::-1] / ((cols * cols + cols) / 2)
        case "bad proposals":
            weights = np.arange(1, cols + 1) / ((cols * cols + cols) / 2)
        case "weight on one":
            one_weight = 0.8
            weights = np.full(cols, (1 - one_weight) / (cols - 1))
            weights[-2] = one_weight
        case _:
            raise NotImplementedError

    return weights @ np.abs(1 - np.cumsum(Q, axis=0)).sum(axis=0)

In [16]:
x = np.array([0, 1, 0, 0, 0]).reshape(-1, 1)
x

array([[0],
       [1],
       [0],
       [0],
       [0]])

In [17]:
wasserstein_transition_distance_to_optimal(x)

np.float64(1.0)

In [18]:
def random_decision_scheme(n: int, seed: int | None = None) -> pl.DataFrame:
    if seed is not None:
        np.random.seed(seed)
    size = n + 1
    data = np.random.rand(size, size)
    normalized = data / data.sum(axis=0, keepdims=True)
    return pl.DataFrame(
        {f"from {i}": normalized[:, i] for i in reversed(range(size))}
    ).select((pl.lit("to ") + pl.row_index().reverse().cast(str)).alias("_"), pl.all())

In [19]:
CASES = {
    "perfect": [
        [1.0, 1.0, 1.0, 1.0],
        [0.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0],
    ],
    "almost perfect": [
        [0.9, 0.9, 0.9, 0.8],
        [0.1, 0.1, 0.1, 0.1],
        [0.0, 0.0, 0.0, 0.1],
        [0.0, 0.0, 0.0, 0.0],
    ],
    "bad": [
        [0.4, 0.3, 0.1, 0.1],
        [0.3, 0.4, 0.3, 0.2],
        [0.2, 0.2, 0.4, 0.3],
        [0.1, 0.1, 0.2, 0.4],
    ],
    "really bad": [
        [0.1, 0.2, 0.1, 0.1],
        [0.3, 0.1, 0.2, 0.2],
        [0.2, 0.3, 0.4, 0.3],
        [0.4, 0.4, 0.3, 0.4],
    ],
    "worst": [
        [0.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0],
        [1.0, 1.0, 1.0, 1.0],
    ],
}

CASES = {
    x: pl.DataFrame([c for i, c in enumerate(y)], orient="row")
    .select((pl.lit("to ") + pl.row_index().reverse().cast(str)).alias("_"), pl.all())
    .rename(
        lambda x: (
            x.replace("column_", "from ")[:-1] + str(len(y) - 1 - int(x[-1]))
            if x != "_"
            else x
        )
    )
    for x, y in CASES.items()
}

CASES["random1"] = random_decision_scheme(3, 42)
CASES["random2"] = random_decision_scheme(3, 64)
CASES["random3"] = random_decision_scheme(3, 90)

CASES

{'perfect': shape: (4, 5)
 ┌──────┬────────┬────────┬────────┬────────┐
 │ _    ┆ from 3 ┆ from 2 ┆ from 1 ┆ from 0 │
 │ ---  ┆ ---    ┆ ---    ┆ ---    ┆ ---    │
 │ str  ┆ f64    ┆ f64    ┆ f64    ┆ f64    │
 ╞══════╪════════╪════════╪════════╪════════╡
 │ to 3 ┆ 1.0    ┆ 1.0    ┆ 1.0    ┆ 1.0    │
 │ to 2 ┆ 0.0    ┆ 0.0    ┆ 0.0    ┆ 0.0    │
 │ to 1 ┆ 0.0    ┆ 0.0    ┆ 0.0    ┆ 0.0    │
 │ to 0 ┆ 0.0    ┆ 0.0    ┆ 0.0    ┆ 0.0    │
 └──────┴────────┴────────┴────────┴────────┘,
 'almost perfect': shape: (4, 5)
 ┌──────┬────────┬────────┬────────┬────────┐
 │ _    ┆ from 3 ┆ from 2 ┆ from 1 ┆ from 0 │
 │ ---  ┆ ---    ┆ ---    ┆ ---    ┆ ---    │
 │ str  ┆ f64    ┆ f64    ┆ f64    ┆ f64    │
 ╞══════╪════════╪════════╪════════╪════════╡
 │ to 3 ┆ 0.9    ┆ 0.9    ┆ 0.9    ┆ 0.8    │
 │ to 2 ┆ 0.1    ┆ 0.1    ┆ 0.1    ┆ 0.1    │
 │ to 1 ┆ 0.0    ┆ 0.0    ┆ 0.0    ┆ 0.1    │
 │ to 0 ┆ 0.0    ┆ 0.0    ┆ 0.0    ┆ 0.0    │
 └──────┴────────┴────────┴────────┴────────┘,
 'bad': shape: (4, 

In [20]:
pl.DataFrame(
    [
        {
            "case": case,
            "bias": "none" if bias is False else bias,
            "kl_div": wasserstein_transition_distance_to_optimal(
                scheme.drop("_").to_numpy(), biased=bias
            ),
        }
        for case, scheme in CASES.items()
        for bias in ["good proposals", False, "bad proposals", "weight on one"]
    ]
).pivot(on="bias", index="case", values="kl_div")

case,good proposals,none,bad proposals,weight on one
str,f64,f64,f64,f64
"""perfect""",0.0,0.0,0.0,0.0
"""almost perfect""",0.12,0.15,0.18,0.113333
"""bad""",1.27,1.45,1.63,1.633333
"""really bad""",1.91,1.925,1.94,1.906667
"""worst""",3.0,3.0,3.0,3.0
"""random1""",1.121904,1.24608,1.370255,1.131475
"""random2""",1.260592,1.31056,1.360528,1.166399
"""random3""",1.355782,1.479217,1.602652,1.680448


### Decision Scheme Comparison

In [21]:
from scipy.stats import binom


def log_likelihood_sds(
    obs: np.ndarray, scheme: Callable[[int], np.ndarray], epsilon: float = 1e-12
) -> float:
    """
    Log-likelihood of observed group decisions under an SDS model.

    Parameters
    ----------
    obs : np.ndarray of shape (N+1, 2)
        obs[i, 0] = count of groups choosing incorrect
        obs[i, 1] = count of groups choosing correct

    scheme : Callable
        function taking N and returning array of shape (N+1,)
        scheme(N)[i] = P(group correct | i correct members)

    epsilon : float
        Small value to clip probabilities to avoid log(0).
    """
    N = obs.shape[0] - 1
    p_correct = scheme(N)
    p_adj = np.clip(p_correct, epsilon, 1 - epsilon)
    k = obs[:, 1]
    n = obs.sum(axis=1)
    # Calculate log-likelihood for each row (i) and sum them
    # binom.logpmf(k, n, p) calculates: log( (n choose k) * p^k * (1-p)^(n-k) )
    row_log_likelihoods = binom.logpmf(k, n, p_adj).sum()

    return float(row_log_likelihoods)


def AIC(
    obs: np.ndarray, scheme: Callable[[int], np.ndarray], epsilon: float = 1e-12
) -> float:
    ll = log_likelihood_sds(obs, scheme, epsilon)
    k = 0
    return 2 * k - 2 * ll

In [22]:
def proportionality(N):
    """
    Probability equals proportion of correct members.
    """
    return np.array([i / N for i in range(N + 1)])

def equiprobability(N):
    """
    always the same probability
    """
    return np.array([0.5 for i in range(N + 1)])

def majority_rule(N):
    """
    Majority rule with tie = 0.5
    """
    scheme = np.zeros(N + 1)
    for i in range(N + 1):
        if i > N / 2:
            scheme[i] = 1.0
        elif i == N / 2:
            scheme[i] = 0.5
        else:
            scheme[i] = 0.0
    return scheme


def truth_wins(N):
    """
    At least one correct member guarantees correct group decision.
    """
    scheme = np.zeros(N + 1)
    scheme[1:] = 1.0
    return scheme


def truth_supported(N):
    """
    At least two correct members required.
    """
    scheme = np.zeros(N + 1)
    if N >= 2:
        scheme[2:] = 1.0
    return scheme


def unanimity(N):
    """
    All members must be correct.
    """
    scheme = np.zeros(N + 1)
    scheme[N] = 1.0
    return scheme


def random_rule(N):
    """
    Random responding (baseline).
    """
    return np.full(N + 1, 0.5)


def incorrect_wins(N):
    """
    Mirror of truth-wins: any incorrect member leads to incorrect decision.
    """
    scheme = np.ones(N + 1)
    scheme[1:] = 0.0
    return scheme


def minority_wins(N):
    """
    Mirror-type rule: group tends toward minority (contrast model).
    """
    scheme = np.ones(N + 1)
    scheme[N] = 0.0
    return scheme


# Optional: convenience dictionary for iteration
SCHEMES = {
    "proportionality": proportionality,
    "majority_rule": majority_rule,
    "truth_wins": truth_wins,
    "truth_supported": truth_supported,
    "unanimity": unanimity,
    "random_rule": random_rule,
    "incorrect_wins": incorrect_wins,
    "minority_wins": minority_wins,
}

In [23]:
observation = np.array([[45, 67, 4, 1, 2], [2, 3, 36, 45, 35]]).transpose()
print("Observation: ", observation)
print("----------")

for name, s in SCHEMES.items():
    ll = log_likelihood_sds(observation, s, epsilon=0.05)
    print(f"{name}", ll)

Observation:  [[45  2]
 [67  3]
 [ 4 36]
 [ 1 45]
 [ 2 35]]
----------
proportionality -41.92903521716731
majority_rule -21.891243723709913
truth_wins -196.438938860132
truth_supported -7.99484419347988
unanimity -231.77220661012936
random_rule -126.70650912485733
incorrect_wins -455.5495690267788
minority_wins -420.21630127678145


# OLD

In [24]:
def get_optimal(N):
    return pl.DataFrame(
        {f"from {i}": [1.0] + [0.0 for i in range(N)] for i in reversed(range(N + 1))}
    ).select((pl.lit("to ") + pl.row_index().reverse().cast(str)).alias("_"), pl.all())

In [25]:
def kl_divergence(
    df_p: pl.DataFrame,
    df_q: pl.DataFrame,
    biased: Literal["good proposals", "bad proposals", False] = False,
) -> float:
    P = df_p.to_numpy()  # shape: (n_targets, n_sources)
    Q = df_q.to_numpy()

    epsilon = 1e-12

    P = np.clip(P, epsilon, 1.0)
    Q = np.clip(Q, epsilon, 1.0)

    # renormalize after clipping
    P /= P.sum(axis=0, keepdims=True)
    Q /= Q.sum(axis=0, keepdims=True)

    kl_per_column = np.sum(P * np.log2(P / Q), axis=0)

    cols = len(kl_per_column)

    match biased:
        case False:
            weights = np.ones(cols) / cols
        case "good proposals":
            weights = np.arange(1, cols + 1)[::-1] / ((cols * cols + cols) / 2)
        case "bad proposals":
            weights = np.arange(1, cols + 1) / ((cols * cols + cols) / 2)
        case _:
            raise NotImplementedError

    return float(np.average(kl_per_column, weights=weights))

In [26]:
from scipy.stats import wasserstein_distance


def wasserstein_transition_distance_old(
    df_p: pl.DataFrame,
    df_q: pl.DataFrame,
    biased: Literal["good proposals", "bad proposals", False] = False,
) -> float:
    if df_p.shape != df_q.shape:
        raise ValueError("Both DataFrames must have the same shape")

    P = df_p.to_numpy()  # shape (n_target_states, n_source_states)
    Q = df_q.to_numpy()

    n_targets, n_sources = P.shape
    positions = np.arange(n_targets, dtype=float)
    total_distance = 0.0

    cols = n_sources

    match biased:
        case False:
            weights = np.ones(cols) / cols
        case "good proposals":
            weights = np.arange(1, cols + 1)[::-1] / ((cols * cols + cols) / 2)
        case "bad proposals":
            weights = np.arange(1, cols + 1) / ((cols * cols + cols) / 2)
        case _:
            raise NotImplementedError

    for j in range(n_sources):
        total_distance += weights[j] * wasserstein_distance(
            positions, positions, u_weights=P[:, j], v_weights=Q[:, j]
        )

    return total_distance


def wasserstein_transition_distance(
    df_p: pl.DataFrame,
    df_q: pl.DataFrame,
    biased: Literal["good proposals", "bad proposals", False] = False,
) -> float:
    if df_p.shape != df_q.shape:
        raise ValueError("Both DataFrames must have the same shape")
    P = df_p.to_numpy()
    Q = df_q.to_numpy()
    rows, cols = P.shape

    match biased:
        case False:
            weights = np.ones(cols) / cols
        case "good proposals":
            weights = np.arange(1, cols + 1)[::-1] / ((cols * cols + cols) / 2)
        case "bad proposals":
            weights = np.arange(1, cols + 1) / ((cols * cols + cols) / 2)
        case _:
            raise NotImplementedError

    return weights @ np.abs(np.cumsum(P, axis=0) - np.cumsum(Q, axis=0)).sum(axis=0)

In [27]:
import numpy as np
from typing import Callable


def log_likelihood_sds(
    obs: np.ndarray, scheme: Callable[[int], np.ndarray], epsilon: float = 1e-12
) -> float:
    """
    Log-likelihood of observed group decisions under an SDS model.

    Parameters
    ----------
    obs : np.ndarray of shape (N+1, 2)
        obs[i, 0] = count of groups choosing correct
        obs[i, 1] = count of groups choosing incorrect

    scheme : Callable
        function taking N and returning array of shape (N+1,)
        scheme[i] = P(group correct | i correct members)

    epsilon : float
        numerical stability to avoid log(0)

    Returns
    -------
    loglik : float
    """

    obs = np.asarray(obs)
    N = obs.shape[0] - 1
    p = np.asarray(scheme(N))

    assert obs.shape[0] == p.shape[0], "Mismatch in rows"
    assert obs.shape[1] == 2, "obs must have 2 columns"

    # Clip probabilities to avoid log(0)
    p = np.clip(p, epsilon, 1 - epsilon)

    loglik = 0.0

    for i in range(obs.shape[0]):
        correct = obs[i, 0]
        incorrect = obs[i, 1]

        if correct + incorrect == 0:
            continue  # skip empty rows

        loglik += correct * np.log(p[i])
        loglik += incorrect * np.log(1 - p[i])

    return float(loglik)


def AIC(
    obs: np.ndarray, scheme: Callable[[int], np.ndarray], epsilon: float = 1e-12
) -> float:
    ll = log_likelihood_sds(obs, scheme, epsilon)
    k = 0
    return 2 * k - 2 * ll

In [28]:
import numpy as np
from scipy.stats import chi2


def sds_chi_square_test(
    obs: np.ndarray, scheme: Callable[[int], np.ndarray], alpha=0.20
):
    """
    Implements the original Social Decision Scheme (SDS) model test.

    Parameters
    ----------
    obs :
        Shape (N+1, 2)
        Columns: [group_incorrect, group_correct]

    scheme :
        function that takes N and returns np.ndarray of shape Shape (N+1,)
        Probability that group is correct given i correct members

    alpha : float
        Significance level (default 0.20 as in SDS literature)

    Returns
    -------
    dict with:
        chi2_stat
        df
        p_value
        reject (True/False)
        expected_counts
    """

    observed_counts = np.asarray(obs)
    decision_scheme_probs = np.asarray(scheme(observed_counts.shape[0] - 1))

    eps = 0.000001
    decision_scheme_probs = np.clip(decision_scheme_probs, 0 + eps, 1 - eps)

    assert observed_counts.shape[0] == decision_scheme_probs.shape[0]
    assert observed_counts.shape[1] == 2

    # total per row
    row_totals = observed_counts.sum(axis=1)

    # expected counts
    expected_correct = row_totals * decision_scheme_probs
    expected_incorrect = row_totals * (1 - decision_scheme_probs)

    expected_counts = np.column_stack([expected_incorrect, expected_correct])

    chi2_stat = np.sum(((observed_counts - expected_counts) ** 2) / expected_counts)

    degrees_of_freedom = observed_counts.shape[0] * (observed_counts.shape[1] - 1)

    p_value = 1 - chi2.cdf(chi2_stat, degrees_of_freedom)

    # SDS interpretation: non-significant = model supported
    reject = p_value < alpha

    return {
        "chi2_stat": float(chi2_stat),
        "degrees_of_freedom": int(degrees_of_freedom),
        "p_value": float(p_value),
        "reject": bool(reject),
        "expected_counts": expected_counts,
    }